# EI Maps

## Preliminary check — reflectance calibration

Before computing any EI map, confirm the hyperspectral cubes hold calibrated reflectance. Calibrated reflectance is expected to lie within (0, 1]; values outside this range would indicate the cubes are not perfectly calibrated.

In [ ]:
from pathlib import Path
import sys

# Locate the project root (holds config.py) regardless of notebook depth.
ROOT = Path.cwd()
while not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew

import config
from src.io_utils import load_cube, load_rgb
from src.manifest import build_manifest

# Reflectance calibration check: calibrated reflectance should lie in (0, 1].
sample = build_manifest(config.DATA_ROOT, config.SPLIT_OVERRIDE).sample(15, random_state=0)
r_min = min(float(load_cube(p).min()) for p in sample["vis_path"])
r_max = max(float(load_cube(p).max()) for p in sample["vis_path"])

print(f"Reflectance range over {len(sample)} sampled cubes: [{r_min:.4f}, {r_max:.4f}]")
if r_min >= 0.0 and r_max <= 1.0 + 1e-6:
    print("Calibrated reflectance in (0, 1], as expected.")
else:
    print("Note: reflectance is not perfectly calibrated — calibrated reflectance is "
          "typically expected within [0, 1].")

## Stage 1 — Verifying raw EI maps

### Statistical results of raw EI maps — no segmentation and preprocessing of the data

The raw EI maps are the unmodified source of truth: computed directly from the hyperspectral cubes, with no skin segmentation and no destriping applied.

- **Plot:** the distribution of all raw EI maps combined, followed by the individual distributions of selected subjects.
- **Metrics:** mean, maximum, minimum, and skewness.
- **Purpose:** establish the initial baseline and capture how heavily skewed the data is due to mixed skin and background pixels.

In [ ]:
# Anchor the (relative) processed path to the project root.
EI_MAPS_DIR = (ROOT / config.LOCAL_PROCESSED_DIR / "ei_maps").resolve()
SELECTED_SUBJECTS = ["p012", "p019", "p027"]


def load_values(files):
    """Flattened EI values concatenated across the given .npy maps."""
    return np.concatenate([np.load(f).ravel() for f in files])


all_files = sorted(EI_MAPS_DIR.glob("*.npy"))
all_values = load_values(all_files)
subject_values = {
    subj: load_values(sorted(EI_MAPS_DIR.glob(f"{subj}_*.npy")))
    for subj in SELECTED_SUBJECTS
}
print(f"Loaded {len(all_files)} raw EI maps ({all_values.size:,} pixels).")

In [ ]:
# This raw stage reports both the true min/max and the clipped (1st/99th percentile)
# range. The clipped range excludes extreme outlier pixels (sensor noise);
# mean and skewness always use all pixels. Later stages report the clipped range only.
ROBUST_PCT = (1, 99)


def describe(values):
    lo, hi = np.percentile(values, ROBUST_PCT)
    return {
        "mean": values.mean(),
        "min": values.min(),
        "max": values.max(),
        "min (1st pct)": lo,
        "max (99th pct)": hi,
        "skewness": skew(values),
    }


metrics = pd.DataFrame(
    {"All maps": describe(all_values),
     **{subj: describe(vals) for subj, vals in subject_values.items()}}
).T.round(2)
metrics

A few extreme pixels — sensor noise rather than skin signal — sit far outside the bulk of the data, which is why the true min/max are so wide. We therefore report and plot the **clipped 1st–99th percentile range**: it reflects where the real values lie and keeps the distributions readable and comparable across the later preprocessing stages. The extremes are excluded only from the view; the mean and skewness are still computed over all pixels.

In [ ]:
lo, hi = np.percentile(all_values, ROBUST_PCT)


def plot_hist(ax, vals, title):
    ax.hist(vals, bins="auto", range=(lo, hi), color="#5a4fcf")
    ax.axvline(vals.mean(), color="black", linestyle="--", label=f"mean = {vals.mean():.1f}")
    ax.axvline(np.median(vals), color="crimson", linestyle=":", label=f"median = {np.median(vals):.1f}")
    ax.set_title(f"{title}  (skew = {skew(vals):+.2f})")
    ax.set_xlabel("EI")
    ax.legend()


fig, ax = plt.subplots(figsize=(9, 4))
plot_hist(ax, all_values, "Pooled Distribution")
ax.set_ylabel("pixel count")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(SELECTED_SUBJECTS), figsize=(15, 4), sharey=True)
for ax, subj in zip(axes, SELECTED_SUBJECTS):
    plot_hist(ax, subject_values[subj], subj)
axes[0].set_ylabel("pixel count")
plt.suptitle("Per-Subject Distribution")
plt.tight_layout()
plt.show()

### Raw EI maps vs. RGB — selected subjects, neutral views

Each RGB image is shown next to its raw EI map (magma), for the three selected subjects across the neutral front, left and right views. Elevated EI (brighter) should fall on visibly red facial regions.

In [ ]:
# RGB image next to its raw EI map (magma) for a subject's neutral views.
# Each panel is titled with the source filename; EI maps share the clipped scale.
NEUTRAL_VIEWS = ["front", "left", "right"]
RGB_DIR = Path(config.DATA_ROOT) / "train" / "RGB"


def show_subject_maps(subj):
    fig, axes = plt.subplots(len(NEUTRAL_VIEWS), 2, figsize=(8, 4 * len(NEUTRAL_VIEWS)))
    for row, view in enumerate(NEUTRAL_VIEWS):
        stem = f"{subj}_neutral_{view}"
        rgb = load_rgb(str(RGB_DIR / f"{stem}.jpg"))
        ei = np.load(EI_MAPS_DIR / f"{stem}.npy")

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"RGB: {stem}.jpg")
        axes[row, 0].axis("off")

        im = axes[row, 1].imshow(ei, cmap="magma", vmin=lo, vmax=hi)
        axes[row, 1].set_title(f"EI map: {stem}.npy")
        axes[row, 1].axis("off")
        fig.colorbar(im, ax=axes[row, 1], fraction=0.046)
    plt.tight_layout()
    plt.show()

In [ ]:
show_subject_maps("p012")

In [ ]:
show_subject_maps("p019")

In [ ]:
show_subject_maps("p027")